# Mushroom Classification — Poisonous vs Non-Poisonous

### Normalization + Decision Tree + Naive Bayes + SVM + KNN

**Goal:** Predict whether a mushroom is **Poisonous** or **Non-Poisonous (Edible)** using its physical/categorical features.

**Models used:**
1. Decision Tree
2. Naive Bayes (CategoricalNB)
3. Support Vector Machine (SVM)
4. K-Nearest Neighbors (KNN)

The notebook also performs categorical encoding and **Min-Max normalization** for the distance-based models.

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder, OneHotEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.naive_bayes import CategoricalNB
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

RANDOM_STATE = 42
sns.set_style("whitegrid")


## 2. Load Dataset

Upload `mushroom.csv` in Colab, or change `DATA_PATH` if your CSV has another name.

In [ ]:
# If using Google Colab, run this cell and select your CSV file.
from google.colab import files

uploaded = files.upload()
DATA_PATH = next((name for name in uploaded.keys() if name.lower().endswith(".csv")), "mushroom.csv")

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
display(df.head())


## 3. Explore the Dataset

In [ ]:
print("Columns:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum())

print("\nClass distribution:")
print(df["Class"].value_counts())


In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(x="Class", data=df, hue="Class", legend=False)
plt.title("Poisonous vs Non-Poisonous (Edible)")
plt.xlabel("Class")
plt.ylabel("Count")
plt.tight_layout()
plt.show()


## 4. Prepare Target: Poisonous vs Non-Poisonous

The original mushroom dataset commonly uses `edible` and `poisonous`.

For this project:
- **Poisonous = 1**
- **Non-Poisonous / Edible = 0**

The code also accepts common variants such as `p/e`, `poisonous/edible`, and `non-poisonous/nonpoisonous`.

In [ ]:
# Clean target labels
df["Class"] = df["Class"].astype(str).str.strip().str.lower()

poisonous_values = {"p", "poisonous", "poison", "1"}
non_poisonous_values = {"e", "edible", "non-poisonous", "nonpoisonous", "non poisonous", "0"}

def map_target(value):
    if value in poisonous_values:
        return 1
    if value in non_poisonous_values:
        return 0
    return np.nan

df["Target"] = df["Class"].map(map_target)

if df["Target"].isna().any():
    print("Unrecognized class values:", df.loc[df["Target"].isna(), "Class"].unique())
    raise ValueError("Please update the target mapping for the class labels in your dataset.")

print(df["Target"].value_counts().rename(index={0: "Non-Poisonous", 1: "Poisonous"}))


## 5. Feature Preparation and Encoding

`SampleID` is only an identifier, so it is removed.

Because the predictors are categorical:
- **Ordinal/label encoding** is used for Decision Tree and Categorical Naive Bayes.
- **One-hot encoding + Min-Max normalization** is used for SVM and KNN. This avoids treating categories as if their integer codes had a real numeric order.

In [ ]:
# Drop ID and target columns
drop_cols = [c for c in ["SampleID", "Class", "Target"] if c in df.columns]
X_raw = df.drop(columns=drop_cols).copy()
y = df["Target"].astype(int)

# Fill any unexpected missing categorical values
X_raw = X_raw.fillna("Unknown").astype(str)

print("Features used:", X_raw.columns.tolist())
print("Number of features:", X_raw.shape[1])


## 6. Train-Test Split

In [ ]:
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw,
    y,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Training samples:", len(X_train_raw))
print("Testing samples :", len(X_test_raw))


## 7. Categorical Encoding for Decision Tree and Naive Bayes

In [ ]:
# Label encode each categorical feature.
# The same category mapping is applied to train and test data.
X_train_cat = pd.DataFrame(index=X_train_raw.index)
X_test_cat = pd.DataFrame(index=X_test_raw.index)

feature_encoders = {}

for col in X_train_raw.columns:
    le = LabelEncoder()
    le.fit(X_train_raw[col])

    X_train_cat[col] = le.transform(X_train_raw[col])

    # Handle a rare unseen test category safely
    known = set(le.classes_)
    X_test_cat[col] = X_test_raw[col].map(
        lambda v: le.transform([v])[0] if v in known else 0
    )

    feature_encoders[col] = le

X_train_cat = X_train_cat.astype(int)
X_test_cat = X_test_cat.astype(int)

print("Encoded training data:")
display(X_train_cat.head())


## 8. Normalization for SVM and KNN

One-hot encoding converts each categorical value into a binary feature.  
Then **Min-Max normalization** scales every feature into the range **0 to 1**.

In [ ]:
onehot = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

X_train_onehot = onehot.fit_transform(X_train_raw)
X_test_onehot = onehot.transform(X_test_raw)

scaler = MinMaxScaler()
X_train_norm = scaler.fit_transform(X_train_onehot)
X_test_norm = scaler.transform(X_test_onehot)

print("One-hot shape:", X_train_onehot.shape)
print("Normalized range:",
      X_train_norm.min(), "to", X_train_norm.max())


## 9. Model 1 — Decision Tree

Decision Tree can work directly with the encoded categorical features.

In [ ]:
dt_model = DecisionTreeClassifier(
    criterion="entropy",
    max_depth=5,
    random_state=RANDOM_STATE
)

dt_model.fit(X_train_cat, y_train)
dt_pred = dt_model.predict(X_test_cat)

print("Decision Tree")
print("Accuracy :", accuracy_score(y_test, dt_pred))
print("Precision:", precision_score(y_test, dt_pred, zero_division=0))
print("Recall   :", recall_score(y_test, dt_pred, zero_division=0))
print("F1-score :", f1_score(y_test, dt_pred, zero_division=0))

print("\nClassification Report:")
print(classification_report(
    y_test, dt_pred,
    target_names=["Non-Poisonous", "Poisonous"],
    zero_division=0
))


In [ ]:
plt.figure(figsize=(18, 9))
plot_tree(
    dt_model,
    feature_names=X_raw.columns,
    class_names=["Non-Poisonous", "Poisonous"],
    filled=True,
    rounded=True,
    fontsize=8,
    max_depth=3
)
plt.title("Decision Tree (Top Levels)")
plt.tight_layout()
plt.show()


## 10. Model 2 — Naive Bayes

`CategoricalNB` is used because the original predictor variables are categorical.

In [ ]:
nb_model = CategoricalNB()

nb_model.fit(X_train_cat, y_train)
nb_pred = nb_model.predict(X_test_cat)

print("Naive Bayes")
print("Accuracy :", accuracy_score(y_test, nb_pred))
print("Precision:", precision_score(y_test, nb_pred, zero_division=0))
print("Recall   :", recall_score(y_test, nb_pred, zero_division=0))
print("F1-score :", f1_score(y_test, nb_pred, zero_division=0))

print("\nClassification Report:")
print(classification_report(
    y_test, nb_pred,
    target_names=["Non-Poisonous", "Poisonous"],
    zero_division=0
))


## 11. Model 3 — Support Vector Machine (SVM)

SVM is trained on the **one-hot encoded and normalized** feature matrix.

In [ ]:
svm_model = SVC(
    kernel="rbf",
    random_state=RANDOM_STATE
)

svm_model.fit(X_train_norm, y_train)
svm_pred = svm_model.predict(X_test_norm)

print("SVM")
print("Accuracy :", accuracy_score(y_test, svm_pred))
print("Precision:", precision_score(y_test, svm_pred, zero_division=0))
print("Recall   :", recall_score(y_test, svm_pred, zero_division=0))
print("F1-score :", f1_score(y_test, svm_pred, zero_division=0))

print("\nClassification Report:")
print(classification_report(
    y_test, svm_pred,
    target_names=["Non-Poisonous", "Poisonous"],
    zero_division=0
))


## 12. Model 4 — K-Nearest Neighbors (KNN)

KNN is also trained on the **normalized** feature matrix because it is distance-based.

In [ ]:
knn_model = KNeighborsClassifier(n_neighbors=5)

knn_model.fit(X_train_norm, y_train)
knn_pred = knn_model.predict(X_test_norm)

print("KNN")
print("Accuracy :", accuracy_score(y_test, knn_pred))
print("Precision:", precision_score(y_test, knn_pred, zero_division=0))
print("Recall   :", recall_score(y_test, knn_pred, zero_division=0))
print("F1-score :", f1_score(y_test, knn_pred, zero_division=0))

print("\nClassification Report:")
print(classification_report(
    y_test, knn_pred,
    target_names=["Non-Poisonous", "Poisonous"],
    zero_division=0
))


## 13. Confusion Matrices

In [ ]:
predictions = {
    "Decision Tree": dt_pred,
    "Naive Bayes": nb_pred,
    "SVM": svm_pred,
    "KNN": knn_pred
}

fig, axes = plt.subplots(2, 2, figsize=(11, 9))

for ax, (name, pred) in zip(axes.ravel(), predictions.items()):
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        ax=ax,
        xticklabels=["Non-Poisonous", "Poisonous"],
        yticklabels=["Non-Poisonous", "Poisonous"]
    )
    ax.set_title(name)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")

plt.tight_layout()
plt.show()


## 14. Model Comparison

In [ ]:
results = []

for name, pred in predictions.items():
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1-Score": f1_score(y_test, pred, zero_division=0)
    })

comparison = pd.DataFrame(results).sort_values("Accuracy", ascending=False).reset_index(drop=True)
display(comparison)


In [ ]:
comparison_plot = comparison.set_index("Model")[
    ["Accuracy", "Precision", "Recall", "F1-Score"]
]

comparison_plot.plot(kind="bar", figsize=(10, 5))
plt.title("Decision Tree vs Naive Bayes vs SVM vs KNN")
plt.ylabel("Score")
plt.ylim(0, 1.05)
plt.xticks(rotation=0)
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()


## 15. Best Model

For this problem, **Recall for the Poisonous class is especially important** because predicting a poisonous mushroom as non-poisonous is the more dangerous error.

The code below automatically reports the best model by accuracy and the best model by poisonous-class recall.

In [ ]:
best_accuracy_model = comparison.loc[comparison["Accuracy"].idxmax(), "Model"]
best_recall_model = comparison.loc[comparison["Recall"].idxmax(), "Model"]

print("Best model by Accuracy:", best_accuracy_model)
print("Best model by Poisonous-class Recall:", best_recall_model)

print("\nFinal comparison:")
display(comparison)


## 16. Conclusion

- The dataset is classified into **Poisonous** and **Non-Poisonous (Edible)** classes.
- Categorical features are encoded before model training.
- **Min-Max normalization** is applied after one-hot encoding for SVM and KNN.
- Four models are compared: **Decision Tree, Naive Bayes, SVM, and KNN**.
- Accuracy, Precision, Recall, F1-score, and confusion matrices are used for evaluation.
- For mushroom safety, poisonous-class **Recall** should be given special importance.

> **Note:** This is an educational ML classification project. A model prediction should not be used as a real-world guarantee that a mushroom is safe to eat.
